# Florida NASA POWER EDA and Dashboard Development
This notebook documents the reproducible EDA workflow for the capstone project.

## Dataset source
Data were downloaded from NASA POWER for five Florida cities from 2015 to present. The final dashboard embeds the cleaned CSV data directly in the HTML.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/florida_nasa_power_daily.csv', parse_dates=['date'])
df.head()

In [ ]:
df.shape, df.columns.tolist(), df.isna().sum()

In [ ]:
metric_cols = ['T2M','T2M_MAX','T2M_MIN','PRECTOTCORR','RH2M','WS2M','ALLSKY_SFC_SW_DWN','PS','T2MDEW']
df[metric_cols].describe().round(2)

## Visualization 1: Monthly temperature trend

In [ ]:
monthly = df.groupby('year_month', as_index=False).agg(T2M=('T2M','mean'), PRECTOTCORR=('PRECTOTCORR','mean'))
monthly['date'] = pd.to_datetime(monthly['year_month'] + '-01')
plt.figure(figsize=(10,4))
plt.plot(monthly['date'], monthly['T2M'])
plt.title('Monthly Average Temperature Trend')
plt.xlabel('Date')
plt.ylabel('Temperature (deg C)')
plt.show()

## Visualization 2: Monthly precipitation trend

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(monthly['date'], monthly['PRECTOTCORR'])
plt.title('Monthly Average Daily Precipitation Trend')
plt.xlabel('Date')
plt.ylabel('Precipitation (mm/day)')
plt.show()

## Visualization 3-4: City comparisons

In [ ]:
city = df.groupby('city').agg(T2M=('T2M','mean'), PRECTOTCORR=('PRECTOTCORR','mean'), RH2M=('RH2M','mean')).sort_values('T2M')
city[['T2M','PRECTOTCORR','RH2M']]

In [ ]:
city['T2M'].plot(kind='barh', title='Average Temperature by City')
plt.xlabel('Temperature (deg C)')
plt.show()
city.sort_values('PRECTOTCORR')['PRECTOTCORR'].plot(kind='barh', title='Average Daily Precipitation by City')
plt.xlabel('Precipitation (mm/day)')
plt.show()

## Visualization 5-6: Heatmaps

In [ ]:
heat = df.groupby(['city','month'])['PRECTOTCORR'].mean().unstack()
plt.figure(figsize=(10,4))
plt.imshow(heat.values, aspect='auto', cmap='turbo')
plt.yticks(range(len(heat.index)), heat.index)
plt.xticks(range(12), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.colorbar(label='Avg precipitation (mm/day)')
plt.title('Seasonal Precipitation Heatmap by City')
plt.show()

## Visualization 7-10: distributions and associations

In [ ]:
df.boxplot(column='T2M', by='city', figsize=(9,4))
plt.suptitle('')
plt.title('Temperature Distribution by City')
plt.ylabel('Temperature (deg C)')
plt.show()

df['PRECTOTCORR'].hist(bins=50, figsize=(9,4))
plt.title('Daily Precipitation Distribution')
plt.xlabel('Precipitation (mm/day)')
plt.ylabel('Daily observations')
plt.show()

In [ ]:
sample = df.sample(min(5000, len(df)), random_state=42)
plt.figure(figsize=(7,5))
plt.scatter(sample['T2M'], sample['RH2M'], s=5, alpha=0.25)
plt.title('Temperature vs Relative Humidity')
plt.xlabel('Temperature (deg C)')
plt.ylabel('Relative humidity (%)')
plt.show()

df[metric_cols].corr().round(2)

## Dashboard development notes
The dashboard was built as a static HTML file. The cleaned CSV was embedded into JavaScript so the file can be shared independently. The CSV remains in the package for storage, transparency, and reproducibility.